# Pet Safe: entrenar el detector de cabezas

Este cuaderno entrena un detector tipo YOLO que encuentra **la cabeza de perros y gatos** en una foto. La app lo usa para recortar justo la cara antes de calcular la huella biométrica.

**Cómo usarlo**
1. Menú **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)** y Guardar.
2. Menú **Entorno de ejecución → Ejecutar todo**.
3. Espera (unos 30 a 60 minutos). Al final se descarga `pet-head.onnx`.
4. Envía ese archivo en el proyecto de Pet Safe (o súbelo a `public/models/` en GitHub).

Datos: Oxford-IIIT Pet (37 razas, cajas de la cabeza hechas a mano). Entrenamiento: Ultralytics YOLO11n (licencia AGPL-3.0).

In [ ]:
!pip -q install ultralytics onnx onnxslim
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO HAY GPU: cambia el tipo de entorno a GPU')

## 1. Descargar las fotos y las cajas de las cabezas

In [ ]:
import os, subprocess
os.makedirs('/content/pets', exist_ok=True)
os.chdir('/content/pets')
FUENTES = ['https://thor.robots.ox.ac.uk/datasets/pets/', 'https://www.robots.ox.ac.uk/~vgg/data/pets/data/']
for archivo in ['images.tar.gz', 'annotations.tar.gz']:
    if os.path.exists(archivo):
        continue
    for base in FUENTES:
        if subprocess.run(['wget', '-q', base + archivo]).returncode == 0:
            break
    else:
        raise SystemExit('No se pudo descargar ' + archivo)
!tar xzf images.tar.gz && tar xzf annotations.tar.gz
print(len(os.listdir('images')), 'fotos,', len(os.listdir('annotations/xmls')), 'cajas')

## 2. Convertir las cajas al formato de YOLO (una clase: cabeza)

In [ ]:
import random, shutil, xml.etree.ElementTree as ET
from pathlib import Path

random.seed(0)
salida = Path('/content/dataset')
xmls = sorted(Path('annotations/xmls').glob('*.xml'))
random.shuffle(xmls)
corte = int(len(xmls) * 0.85)
for parte, lista in [('train', xmls[:corte]), ('val', xmls[corte:])]:
    (salida / 'images' / parte).mkdir(parents=True, exist_ok=True)
    (salida / 'labels' / parte).mkdir(parents=True, exist_ok=True)
    for x in lista:
        raiz = ET.parse(x).getroot()
        nombre = raiz.findtext('filename')
        foto = Path('images') / nombre
        if not foto.exists():
            continue
        W = float(raiz.findtext('size/width')); H = float(raiz.findtext('size/height'))
        lineas = []
        for obj in raiz.iter('object'):
            b = obj.find('bndbox')
            x1, y1, x2, y2 = (float(b.findtext(k)) for k in ('xmin', 'ymin', 'xmax', 'ymax'))
            lineas.append(f'0 {(x1 + x2) / 2 / W:.6f} {(y1 + y2) / 2 / H:.6f} {(x2 - x1) / W:.6f} {(y2 - y1) / H:.6f}')
        shutil.copy(foto, salida / 'images' / parte / nombre)
        (salida / 'labels' / parte / (Path(nombre).stem + '.txt')).write_text('\n'.join(lineas))
(salida / 'data.yaml').write_text(f'path: {salida}\ntrain: images/train\nval: images/val\nnames:\n  0: cabeza\n')
print('Entrenamiento:', len(list((salida / 'labels/train').glob('*.txt'))), ' Validación:', len(list((salida / 'labels/val').glob('*.txt'))))

## 3. Entrenar

In [ ]:
from ultralytics import YOLO
modelo = YOLO('yolo11n.pt')
modelo.train(data='/content/dataset/data.yaml', imgsz=320, epochs=60, batch=64, patience=15,
             fliplr=0.5, degrees=10, scale=0.5, project='/content/runs', name='cabeza', exist_ok=True)

## 4. Revisar qué tan bien funciona

`mAP50` cerca de 0.95 o más es muy bueno. Abajo se ven algunas fotos de validación con la caja encontrada.

In [ ]:
mejor = YOLO('/content/runs/cabeza/weights/best.pt')
m = mejor.val(data='/content/dataset/data.yaml', imgsz=320)
print('mAP50:', round(m.box.map50, 3), ' mAP50-95:', round(m.box.map, 3))
from IPython.display import Image, display
fotos = sorted(Path('/content/dataset/images/val').glob('*.jpg'))[:6]
for r in mejor.predict(fotos, imgsz=320, conf=0.4, save=True, project='/content/runs', name='prueba', exist_ok=True):
    pass
for f in sorted(Path('/content/runs/prueba').glob('*.jpg'))[:6]:
    display(Image(filename=str(f), width=280))

## 5. Exportar para la app y descargar

El archivo `pet-head.onnx` pesa unos 10 MB. La app lo espera con entrada de 320x320 y una sola clase.

In [ ]:
ruta = mejor.export(format='onnx', imgsz=320, opset=12, simplify=True, dynamic=False)
shutil.copy(ruta, '/content/pet-head.onnx')
import onnx
m = onnx.load('/content/pet-head.onnx')
print('Entrada:', [d.dim_value for d in m.graph.input[0].type.tensor_type.shape.dim])
print('Salida:', [d.dim_value for d in m.graph.output[0].type.tensor_type.shape.dim], '(debe ser [1, 5, 2100])')
from google.colab import files
files.download('/content/pet-head.onnx')